# PubMed Document Retrieval & Cleaning Pipeline

---
## 📂 Step 1: Imports + Reproducibility

In [1]:
import os
import json
import math
import time
import random
import re
import unicodedata
import requests
import numpy as np
import torch
import ftfy
from tqdm import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
from dotenv import load_dotenv

load_dotenv()

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
DetectorFactory.seed = 0

# NCBI API key
NCBI_API_KEY = '0c02b6e7f8e56927b51f9421816475b02e08'
RATE_LIMIT = 0.11 if NCBI_API_KEY else 0.34
BATCH_SIZE = 20

DATA_DIR = os.environ.get('DATA_DIR', '/workspace/data')

print(f'SEED: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch version: {torch.__version__}')
print(f'NCBI API key: {"✅ found" if NCBI_API_KEY else "❌ not set"}')
print(f'Data dir: {DATA_DIR}')


SEED: 42
CUDA available: True
PyTorch version: 2.10.0+cu128
NCBI API key: ✅ found
Data dir: /workspace/data


---
## 📂 Step 2: Load Data

In [2]:

os.environ['DATA_DIR'] = '/workspace/data'
data_dir = os.environ.get('DATA_DIR', './data')
DATASET_REGISTRY = {}

# Only load these datasets
ALLOWED_DATASETS = {'13B1_golden', '13B2_golden', '13B3_golden', '13B4_golden', 'training13b'}

print('📂 Loading standard JSON files...')
if not os.path.isdir(data_dir):
    print(f'❌ Directory not found: {data_dir}')
else:
    for filename in sorted(os.listdir(data_dir)):
        if filename.endswith('.json'):
            dataset_name = os.path.splitext(filename)[0]

            if dataset_name not in ALLOWED_DATASETS:  # ← filter here
                print(f'⏭️  Skipping: {filename}')
                continue

            file_path = os.path.join(data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    raw_data = json.load(f)
                    if isinstance(raw_data, dict) and 'questions' in raw_data:
                        data_list = raw_data['questions']
                    elif isinstance(raw_data, list):
                        data_list = raw_data
                    else:
                        data_list = [raw_data]
                    DATASET_REGISTRY[dataset_name] = {'data': data_list}
                except json.JSONDecodeError as e:
                    print(f'⚠️ Could not parse {filename}: {e}')

print('\n📋 Dataset Inventory:')
for name, meta in DATASET_REGISTRY.items():
    print(f'  [{name}] — {len(meta["data"])} documents')

📂 Loading standard JSON files...

📋 Dataset Inventory:
  [13B1_golden] — 85 documents
  [13B2_golden] — 85 documents
  [13B3_golden] — 85 documents
  [13B4_golden] — 85 documents
  [training13b] — 5389 documents


---
## 📂 Step 3: Cleaning Functions

In [3]:
import re
import ftfy
from langdetect import detect, LangDetectException
from tqdm import tqdm

def remove_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    return ftfy.fix_text(text)

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = 'en') -> bool:
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text

def strip_pubmed_metadata(text: str) -> str:
    # Extract abstract body — starts after section headers
    # e.g. INTRODUCTION:, BACKGROUND:, OBJECTIVE:, METHODS:, RESULTS:, ABSTRACT:
    abstract_match = re.search(
        r'(INTRODUCTION|BACKGROUND|OBJECTIVE|OBJECTIVES|SUMMARY|ABSTRACT|PURPOSE|AIM|AIMS)\s*:',
        text, flags=re.IGNORECASE
    )
    if abstract_match:
        text = text[abstract_match.start():]  # ← keep everything from here
    else:
        # Fallback: remove first 3 lines (citation + authors)
        lines = text.strip().splitlines()
        text = '\n'.join(lines[3:])

    # Remove remaining metadata
    text = re.sub(r'\b(doi|epub|pmid|pubmed)\b.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '', text)
    text = re.sub(r'.*\b(university|hospital|department|institute|college|school|centre|center)\b.*',
                  '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text(text: str) -> str | None:
    """Full cleaning pipeline for fetched abstract text."""
    text = strip_pubmed_metadata(text)  # ← add as first step
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return text

def clean_cached_abstract(text: str) -> str | None:
    """Faster cleaning for PubMed abstracts — skips language detection."""
    text = strip_pubmed_metadata(text)
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    text = remove_pii(text)
    return text

def clean_document(doc: dict) -> dict | None:
    text = doc.get('body', '')
    if not text and 'snippets' in doc:
        text = ' '.join([s.get('text', '') for s in doc['snippets']])
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return {**doc, 'text': text}

# Apply cleaning
cleaned_datasets = {}
for name, meta in DATASET_REGISTRY.items():
    print(f'🧹 Cleaning dataset: {name} ({len(meta["data"])} documents)')
    cleaned_docs = []
    for doc in tqdm(meta['data'], desc=f'Cleaning {name}'):
        cleaned_doc = clean_document(doc)
        if cleaned_doc:
            cleaned_docs.append(cleaned_doc)
    cleaned_datasets[name] = cleaned_docs
    print(f'✅ Cleaned {len(cleaned_docs)} documents from {name}')

🧹 Cleaning dataset: 13B1_golden (85 documents)


Cleaning 13B1_golden: 100% 85/85 [00:00<00:00, 155.59it/s]


✅ Cleaned 73 documents from 13B1_golden
🧹 Cleaning dataset: 13B2_golden (85 documents)


Cleaning 13B2_golden: 100% 85/85 [00:00<00:00, 330.10it/s]


✅ Cleaned 75 documents from 13B2_golden
🧹 Cleaning dataset: 13B3_golden (85 documents)


...

✅ Cleaned 76 documents from 13B3_golden
🧹 Cleaning dataset: 13B4_golden (85 documents)


Cleaning 13B4_golden: 100% 85/85 [00:00<00:00, 304.01it/s]


✅ Cleaned 70 documents from 13B4_golden
🧹 Cleaning dataset: training13b (5389 documents)


Cleaning training13b: 100% 5389/5389 [00:12<00:00, 431.90it/s]

✅ Cleaned 4804 documents from training13b


In [4]:
def avg_word_length(text: str) -> float:
    words = text.split()
    return sum(len(w) for w in words) / max(len(words), 1)

def bullet_density(text: str) -> float:
    lines = text.splitlines()
    bullet_lines = sum(1 for l in lines if l.strip().startswith(('•', '-', '*', '·')))
    return bullet_lines / max(len(lines), 1)

def passes_quality_filter(doc: dict, verbose: bool = False) -> bool:
    text = doc.get('text', '')
    if not text or len(text) < 20:
        if verbose: print(f'FAIL length: {len(text)}')
        return False
    awl = avg_word_length(text)
    if not (2.5 <= awl <= 15):
        if verbose: print(f'FAIL awl: {awl:.2f}')
        return False
    if bullet_density(text) > 0.6:
        if verbose: print('FAIL bullets')
        return False
    return True

quality_filtered = {}
for name, docs in cleaned_datasets.items():
    before = len(docs)
    filtered = [doc for doc in docs if passes_quality_filter(doc)]
    quality_filtered[name] = filtered
    print(f'[{name}] quality filter: {before} → {len(filtered)} docs')

print('\n✅ Quality filter complete')


[13B1_golden] quality filter: 73 → 73 docs
[13B2_golden] quality filter: 75 → 75 docs
[13B3_golden] quality filter: 76 → 76 docs
[13B4_golden] quality filter: 70 → 70 docs
[training13b] quality filter: 4804 → 4803 docs

✅ Quality filter complete


In [5]:
import unicodedata

def normalize_text(text: str, form: str = 'NFC') -> str:
    text = unicodedata.normalize(form, text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\t', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    return text

normalized_datasets = {}
for name, docs in quality_filtered.items():
    normalized = [{**doc, 'text': normalize_text(doc['text'])} for doc in docs]
    normalized_datasets[name] = normalized

training_raw_data = normalized_datasets['training13b']
test_raw_data = (
    normalized_datasets['13B1_golden'] +
    normalized_datasets['13B2_golden'] +
    normalized_datasets['13B3_golden'] +
    normalized_datasets['13B4_golden']
)

print('✅ Normalization complete')
print(f'Training docs: {len(training_raw_data)}')
print(f'Test docs:     {len(test_raw_data)}')
for name, docs in normalized_datasets.items():
    if docs:
        sample = docs[0]['text']
        print(f'\n[{name}] sample: {sample[:120]}...' if len(sample) > 120 else f'\n[{name}] sample: {sample}')

✅ Normalization complete
Training docs: 4803
Test docs:     294

[13B1_golden] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[13B2_golden] sample: Which ensemble machine-learning framework has been developed harnessing UK biobank data?

[13B3_golden] sample: How many primary genetic associations were identified through pQTL mapping within the Pharma Proteomics Project?

[13B4_golden] sample: Should Zotiraciclib be used for glioblastoma?

[training13b] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?


---
## 📂 Step 4: Fetching documents

In [6]:
BATCH_SIZE = 20  # fetch 20 abstracts per request

def extract_pmid(url: str) -> str:
    """Extract PMID from PubMed URL."""
    return url.rstrip('/').split('/')[-1]

def parse_batch_response(response_text: str, pmids: list) -> dict:
    """Parse multi-abstract response and map back to PMIDs."""
    results = {}
    records = re.split(r'\n\n(?=\d+\.)', response_text)
    for record in records:
        record = record.strip()
        if not record:
            continue
        pmid_match = re.search(r'PMID:\s*(\d+)', record)
        if pmid_match:
            pmid = pmid_match.group(1)
            if pmid in pmids:
                results[pmid] = record
    return results

def fetch_pubmed_batch(urls: list, api_key: str = None) -> dict:
    """Fetch multiple abstracts in one API call. Returns {url: abstract_text}."""
    try:
        url_to_pmid = {}
        for url in urls:
            pmid = extract_pmid(url)
            if pmid.isdigit():
                url_to_pmid[url] = pmid
        if not url_to_pmid:
            return {}
        pmids = list(url_to_pmid.values())
        params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'rettype': 'abstract',
            'retmode': 'text'
        }
        if api_key:
            params['api_key'] = api_key
        response = requests.get(
            'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi',
            params=params,
            timeout=30
        )
        if response.status_code != 200:
            return {}
        pmid_to_text = parse_batch_response(response.text, pmids)
        return {
            url: pmid_to_text.get(pmid, response.text)
            for url, pmid in url_to_pmid.items()
        }
    except Exception as e:
        print(f'Batch fetch error: {e}')
        return {}

# Test with first 3 URLs
test_urls = [s.get('document', '') for s in training_raw_data[0].get('snippets', [])[:3]]
print(f'Testing batch fetch with {len(test_urls)} URLs...')
results = fetch_pubmed_batch(test_urls, NCBI_API_KEY)
for url, text in results.items():
    print(f'URL: {url}')
    print(f'Abstract ({len(text)} chars): {text[:200]}...')


Testing batch fetch with 3 URLs...
URL: http://www.ncbi.nlm.nih.gov/pubmed/15829955
Abstract (1639 chars): 1. Nature. 2005 Apr 14;434(7035):857-63. doi: 10.1038/nature03467.

A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease 
risk.

Emison ES(1), McCallion AS, Kashuk CS, Bush...
URL: http://www.ncbi.nlm.nih.gov/pubmed/15617541
Abstract (1604 chars): 2. Clin Genet. 2005 Jan;67(1):6-14. doi: 10.1111/j.1399-0004.2004.00319.x.

Studying the genetics of Hirschsprung's disease: unraveling an oligogenic 
disorder.

Brooks AS(1), Oostra BA, Hofstra RM.

...
URL: http://www.ncbi.nlm.nih.gov/pubmed/12239580
Abstract (1611 chars): 3. Int J Mol Med. 2002 Oct;10(4):367-70.

Hirschsprung, RET-SOX and beyond: the challenge of examining non-mendelian 
traits (Review).

Pusch CM(1), Sasiadek MM, Blin N.

Author information:
(1)Divisi...


In [7]:
# Deduplicate URLs (avoid redundant fetches)
# Collect all unique PubMed URLs across all docs
all_urls = set()
for doc in training_raw_data:
    for snippet in doc.get('snippets', []):
        url = snippet.get('document', '')
        if url:
            all_urls.add(url)

print(f'Total docs:        {len(training_raw_data)}')
print(f'Unique PubMed URLs: {len(all_urls)}')

# Estimate time
est_seconds = len(all_urls) * RATE_LIMIT
est_minutes = est_seconds / 60
print(f'\nEstimated fetch time: {est_minutes:.1f} minutes')
print(f'(Based on {RATE_LIMIT}s/request × {len(all_urls)} unique URLs)')

Total docs:        4803
Unique PubMed URLs: 37596

Estimated fetch time: 68.9 minutes
(Based on 0.11s/request × 37596 unique URLs)


In [8]:
# Load Cache (skip re-fetching on reruns) ──────────────────
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')

if os.path.exists(cache_path):
    with open(cache_path, 'r', encoding='utf-8') as f:
        abstract_cache = json.load(f)
    print(f'✅ Loaded {len(abstract_cache)} cached abstracts')
else:
    print('❌ No cache found — run next cell first')

❌ No cache found — run next cell first


In [9]:
import math

abstract_cache = {}
failed_urls = []
all_url_list = list(all_urls)
n_batches = math.ceil(len(all_url_list) / BATCH_SIZE)

print(f'Fetching {len(all_url_list)} URLs in {n_batches} batches of {BATCH_SIZE}...')

for i in tqdm(range(0, len(all_url_list), BATCH_SIZE), desc='Fetching batches', total=n_batches):
    batch = all_url_list[i:i + BATCH_SIZE]
    results = fetch_pubmed_batch(batch, NCBI_API_KEY)
    
    for url in batch:
        if url in results and results[url]:
            abstract_cache[url] = results[url]
        else:
            failed_urls.append(url)
    
    time.sleep(RATE_LIMIT)  # rate limit per batch not per URL

print(f'\n✅ Fetched:  {len(abstract_cache)} abstracts')
print(f'❌ Failed:   {len(failed_urls)} URLs')

# Estimate actual speedup
print(f'\nBatches made: {n_batches} (vs {len(all_url_list)} individual requests without batching)')

# Save cache
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')
with open(cache_path, 'w', encoding='utf-8') as f:
    json.dump(abstract_cache, f, indent=2, ensure_ascii=False)
print(f'✅ Cache saved to {cache_path}')


Fetching 37596 URLs in 1880 batches of 20...


Fetching batches: 100% 1880/1880 [41:18<00:00,  1.32s/it]



✅ Fetched:  37576 abstracts
❌ Failed:   20 URLs

Batches made: 1880 (vs 37596 individual requests without batching)
✅ Cache saved to /workspace/data/pubmed_abstract_cache.json


In [11]:
# Pre-clean entire cache once
print("Pre-cleaning abstracts...")
cleaned_cache = {}
for url, text in tqdm(abstract_cache.items(), desc="Cleaning cache"):
    cleaned = clean_cached_abstract(text)
    if cleaned:
        cleaned_cache[url] = cleaned
print(f"✅ {len(cleaned_cache)} abstracts cleaned")

Pre-cleaning abstracts...


Cleaning cache: 100% 37576/37576 [01:39<00:00, 376.38it/s] 

✅ 37572 abstracts cleaned


---
## 📂 Step 5: Build enriched training data

In [12]:
enriched_docs = []
skipped = 0

for doc in tqdm(training_raw_data, desc='Building enriched docs'):
    question = doc.get('body', '')
    snippets = doc.get('snippets', [])

    full_texts = []
    source_urls = []
    pmids = []
    seen_urls = set()

    for snippet in snippets:
        url = snippet.get('document', '')
        if url in seen_urls:
            continue
        seen_urls.add(url)

        if url in cleaned_cache:
            full_texts.append(cleaned_cache[url])
            source_urls.append(url)
            pmids.append(extract_pmid(url))
        else:
            fallback = clean_cached_abstract(snippet.get('text', ''))
            if fallback:
                full_texts.append(fallback)
                source_urls.append(url)
                pmids.append(extract_pmid(url))

    if not full_texts:
        skipped += 1
        continue

    enriched_docs.append({
        **doc,
        "text": " ".join(full_texts),  # ← joined for classification
        "question": question,
        "source_urls": source_urls,
        "pmids": pmids,
        "n_sources": len(source_urls)
    })

print(f'✅ Enriched docs: {len(enriched_docs)}')
print(f'⚠️  Skipped: {skipped}')

Building enriched docs: 100% 4803/4803 [00:00<00:00, 15829.74it/s]

✅ Enriched docs: 4801
⚠️  Skipped: 2


---
## 📂 Step 6: Zero-shot Categorization on Full Abstracts

In [14]:
from transformers import pipeline
from tqdm import tqdm
from collections import Counter
import torch
import gc

CATEGORIES = [
    "Genetics & Mutations",
    "Cancer & Oncology",
    "Pharmacology & Drugs",
    "Neurology & Brain",
    "Cardiology & Heart",
    "Infectious Disease",
    "Immunology",
    "Metabolism & Diabetes",
    "Rare Diseases",
    "Surgery & Procedures",
    "Diagnostics & Imaging",
    "Mental Health",
    "Molecular Biology",
    "Clinical Guidelines",
    "Other"
]

device = 0 if torch.cuda.is_available() else -1
print(f'Using device: {"cuda" if device == 0 else "cpu"}')

print("Loading classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)
print("✅ Classifier ready!")

# Classify all docs
for doc in tqdm(enriched_docs, desc="Classifying"):
    result = classifier(doc['text'][:256], CATEGORIES)
    doc['category'] = result['labels'][0]
    doc['confidence'] = round(result['scores'][0], 4)
    doc['topic_id'] = CATEGORIES.index(result['labels'][0])

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

cats = Counter(doc['category'] for doc in enriched_docs)
print(f'\n✅ Done! {len(cats)} categories')
print(f'{"Category":<40} {"Count":>6} {"%":>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / len(enriched_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading classifier...


Loading weights: 100% 106/106 [00:00<00:00, 560.71it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Classifier ready!


Classifying:   0% 9/4801 [00:02<13:00,  6.14it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Classifying: 100% 4801/4801 [07:41<00:00, 10.41it/s]



✅ Done! 15 categories
Category                                  Count      %
-------------------------------------------------------
Other                                       770 (16.0%)
Pharmacology & Drugs                        703 (14.6%)
Molecular Biology                           598 (12.5%)
Rare Diseases                               474 (9.9%)
Cancer & Oncology                           404 (8.4%)
Genetics & Mutations                        394 (8.2%)
Infectious Disease                          372 (7.7%)
Immunology                                  365 (7.6%)
Cardiology & Heart                          252 (5.2%)
Surgery & Procedures                        173 (3.6%)
Neurology & Brain                           132 (2.7%)
Metabolism & Diabetes                        50 (1.0%)
Mental Health                                50 (1.0%)
Diagnostics & Imaging                        46 (1.0%)
Clinical Guidelines                          18 (0.4%)


In [15]:
# Explode enriched_docs to one row per abstract
exploded_docs = []
for doc in enriched_docs:
    for pmid, text, url in zip(doc['pmids'], 
                                [cleaned_cache.get(u, '') for u in doc['source_urls']], 
                                doc['source_urls']):
        exploded_docs.append({
            'question':   doc['question'],
            'type':       doc.get('type', ''),
            'text':       text,              # ← single abstract
            'category':   doc['category'],   # ← from joined classification
            'topic_id':   doc.get('topic_id', ''),
            'confidence': doc.get('confidence', ''),
            'pmid':       pmid,
            'source_url': url,
            'n_sources':  doc['n_sources']
        })

print(f'✅ Exploded docs: {len(exploded_docs)}')

✅ Exploded docs: 39418


In [17]:
from collections import Counter

pmid_groups = {}
for doc in exploded_docs:
    pmid = doc['pmid']
    if pmid not in pmid_groups:
        pmid_groups[pmid] = {
            'pmid': pmid,
            'text': doc['text'],
            'source_url': doc['source_url'],
            'categories': []
        }
    pmid_groups[pmid]['categories'].append(doc['category'])

vector_db_docs = []
for pmid, data in pmid_groups.items():
    most_common_category = Counter(data['categories']).most_common(1)[0][0]
    vector_db_docs.append({
        'pmid': pmid,
        'text': data['text'],
        'source_url': data['source_url'],
        'category': most_common_category
    })

print(f'✅ Unique PMIDs for vector DB: {len(vector_db_docs)}')

✅ Unique PMIDs for vector DB: 37573


In [18]:
from collections import Counter

# Question level
cats = Counter(doc['category'] for doc in enriched_docs)
total = len(enriched_docs)
print(f'=== Question Level ({total} docs) ===')
print(f'{"Category":<40} {"Count":>6} {"%" :>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / total * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

# Abstract level
print(f'\n=== Abstract Level ({len(exploded_docs)} rows) ===')
cats2 = Counter(doc['category'] for doc in exploded_docs)
for cat, count in cats2.most_common():
    pct = count / len(exploded_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

# Vector DB level
print(f'\n=== Vector DB Level ({len(vector_db_docs)} unique PMIDs) ===')
cats3 = Counter(doc['category'] for doc in vector_db_docs)
for cat, count in cats3.most_common():
    pct = count / len(vector_db_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

=== Question Level (4801 docs) ===
Category                                  Count      %
-------------------------------------------------------
Other                                       770 (16.0%)
Pharmacology & Drugs                        703 (14.6%)
Molecular Biology                           598 (12.5%)
Rare Diseases                               474 (9.9%)
Cancer & Oncology                           404 (8.4%)
Genetics & Mutations                        394 (8.2%)
Infectious Disease                          372 (7.7%)
Immunology                                  365 (7.6%)
Cardiology & Heart                          252 (5.2%)
Surgery & Procedures                        173 (3.6%)
Neurology & Brain                           132 (2.7%)
Metabolism & Diabetes                        50 (1.0%)
Mental Health                                50 (1.0%)
Diagnostics & Imaging                        46 (1.0%)
Clinical Guidelines                          18 (0.4%)

=== Abstract Level (39418

---
## 📂 Step 7: Save final data in CSV and Json format

In [27]:
# Save Final Data 
import json
import pandas as pd
import os

# Create output directory
parsed_docs_dir = os.path.join(DATA_DIR, 'parsed_docs')
os.makedirs(parsed_docs_dir, exist_ok=True)

# Save as JSON — keep enriched_docs (question level)
output_train = os.path.join(DATA_DIR, 'training_final.json')
with open(output_train, 'w', encoding='utf-8') as f:
    json.dump(enriched_docs, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(enriched_docs)} training docs to {output_train}')

# Save as CSV — use exploded_docs (one row per abstract)
csv_path = os.path.join(parsed_docs_dir, 'training_final.csv')
df = pd.DataFrame([{
    'question':   doc.get('question', ''),
    'type':       doc.get('type', ''),
    'text':       doc.get('text', ''),
    'category':   doc.get('category', ''),
    'topic_id':   doc.get('topic_id', ''),
    'confidence': doc.get('confidence', ''),
    'pmid':       doc.get('pmid', ''),
    'source_url': doc.get('source_url', ''),
    'n_sources':  doc.get('n_sources', 0),
} for doc in exploded_docs])  # ← exploded_docs not enriched_docs
df.to_csv(csv_path, index=False, encoding='utf-8', quoting=1)
print(f'✅ Saved CSV to {csv_path}')
print(f'   Rows:    {len(df)}')

# Verify
with open(output_train, 'r', encoding='utf-8') as f:
    verify = json.load(f)
print(f'\n✅ Verified JSON: {len(verify)} docs')
print(f'   Fields:   {list(verify[0].keys())}')
print(f'   Category: {verify[0].get("category")}')
print(f'   Sources:  {verify[0].get("n_sources")}')
print(f'   Text:     {verify[0]["text"][:150]}...')

✅ Saved 4801 training docs to /workspace/data/training_final.json
✅ Saved CSV to /workspace/data/parsed_docs/training_final.csv
   Rows:    39418

✅ Verified JSON: 4801 docs
   Fields:   ['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets', 'text', 'question', 'source_urls', 'pmids', 'n_sources', 'category', 'confidence', 'topic_id']
   Category: Other
   Sources:  9
   Text:     risk. Emison ES(1), McCallion AS, Kashuk CS, Bush RT, Grice E, Lin S, Portnoy ME, Cutler DJ, Green ED, Chakravarti A. Author information: The identifi...


In [24]:
# Create output directory
import json
import pandas as pd
import os
parsed_docs_dir = os.path.join(DATA_DIR, 'parsed_docs')
os.makedirs(parsed_docs_dir, exist_ok=True)

# Save test as JSON — question level
output_test = os.path.join(DATA_DIR, 'test_final.json')
with open(output_test, 'w', encoding='utf-8') as f:
    json.dump(test_raw_data, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(test_raw_data)} test docs to {output_test}')

# Save test as CSV — one row per abstract
test_csv_path = os.path.join(parsed_docs_dir, 'test_final.csv')
df_test = pd.DataFrame([{
    'question':   doc.get('question', ''),
    'type':       doc.get('type', ''),
    'text':       doc.get('text', ''),
    'category':   doc.get('category', ''),
    'topic_id':   doc.get('topic_id', ''),
    'confidence': doc.get('confidence', ''),
    'pmid':       doc.get('pmid', ''),
    'source_url': doc.get('source_url', ''),
    'n_sources':  doc.get('n_sources', 0),
} for doc in test_exploded_docs])  # ← test_exploded_docs
df_test.to_csv(test_csv_path, index=False, encoding='utf-8', quoting=1)
print(f'✅ Saved test CSV to {test_csv_path}')
print(f'   Rows:    {len(df_test)}')

✅ Saved 294 test docs to /workspace/data/test_final.json
✅ Saved test CSV to /workspace/data/parsed_docs/test_final.csv
   Rows:    294


In [28]:
import pandas as pd

pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('/workspace/data/parsed_docs/training_final.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (39418, 9)


,question,type,text,category,topic_id,confidence,pmid,source_url,n_sources
0,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"risk. Emison ES(1), McCallion AS, Kashuk CS, Bush RT, Grice E, Lin S, Portnoy ME, Cutler DJ, Gre...",Other,14,0.2069,15829955,http://www.ncbi.nlm.nih.gov/pubmed/15829955,9
1,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"disorder. Brooks AS(1), Oostra BA, Hofstra RM. Author information: Hirschsprung's disease is cha...",Other,14,0.2069,15617541,http://www.ncbi.nlm.nih.gov/pubmed/15617541,9
2,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"traits (Review). Pusch CM(1), Sasiadek MM, Blin N. Author information: Germany. Hirschsprung dis...",Other,14,0.2069,12239580,http://www.ncbi.nlm.nih.gov/pubmed/12239580,9
3,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"Jarmas AL, Weaver DD, Padilla LM, Stecker E, Bender HA. We describe an infant with Hirschsprung ...",Other,14,0.2069,6650562,http://www.ncbi.nlm.nih.gov/pubmed/6650562,9
4,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"mutations to multifactorial Hirschsprung disease liability. Emison ES(1), Garcia-Barcelo M, Gric...",Other,14,0.2069,20598273,http://www.ncbi.nlm.nih.gov/pubmed/20598273,9


In [22]:
print(exploded_docs[0].keys())
print(f"pmid: {exploded_docs[0].get('pmid', 'MISSING')}")
print(f"source_url: {exploded_docs[0].get('source_url', 'MISSING')}")
print(f"question: {exploded_docs[0].get('question', 'MISSING')[:100]}")

dict_keys(['question', 'type', 'text', 'category', 'topic_id', 'confidence', 'pmid', 'source_url', 'n_sources'])
pmid: 15829955
source_url: http://www.ncbi.nlm.nih.gov/pubmed/15829955
question: Is Hirschsprung disease a mendelian or a multifactorial disorder?
